In [1]:
import os
import glob
import numpy as np
import librosa
import math
from sklearn2c import KNNClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ==========================================
# CONFIGURATION (MUST MATCH C CODE)
# ==========================================
DATASET_PATH = "recordings"
EXPORT_DIR = "exported_models"
EXPORT_NAME = "knn_mfcc_config"

SAMPLE_RATE = 8000
FFT_LEN = 1024
HOP_LEN = 512
NUM_MEL_FILTERS = 20
NUM_DCT_COEFFS = 13
NUM_SAMPLES_TO_KEEP = 500 # Flash memory limit

# ==========================================
# C-STYLE FEATURE EXTRACTION (PYTHON PORT)
# ==========================================
def freq_to_mel(f): return 2595.0 * np.log10(1.0 + f / 700.0)
def mel_to_freq(m): return 700.0 * (10.0**(m / 2595.0) - 1.0)

def get_filterbank(sr, n_fft, n_mels):
    # Matches create_mel_fbank in C
    fmin_mel = freq_to_mel(20)
    fmax_mel = freq_to_mel(4000) # C code used 4000 hardcoded
    
    mel_points = np.linspace(fmin_mel, fmax_mel, n_mels + 2)
    hz_points = mel_to_freq(mel_points)
    bin_points = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    
    filters = np.zeros((n_mels, n_fft // 2 + 1))
    
    for i in range(n_mels):
        for j in range(bin_points[i], bin_points[i+1]):
            filters[i, j] = (j - bin_points[i]) / (bin_points[i+1] - bin_points[i])
        for j in range(bin_points[i+1], bin_points[i+2]):
            filters[i, j] = (bin_points[i+2] - j) / (bin_points[i+2] - bin_points[i+1])
            
    return filters

def get_dct_matrix(n_dct, n_mels):
    # Matches create_dct_matrix in C
    matrix = np.zeros((n_dct, n_mels))
    normalizer = np.sqrt(2.0 / n_mels)
    for k in range(n_dct):
        for n in range(n_mels):
            matrix[k, n] = normalizer * np.cos(math.pi / n_mels * (n + 0.5) * k)
    return matrix

# Pre-compute tables once
DCT_MAT = get_dct_matrix(NUM_DCT_COEFFS, NUM_MEL_FILTERS)
MEL_FILTERS = get_filterbank(SAMPLE_RATE, FFT_LEN, NUM_MEL_FILTERS)
WINDOW = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(FFT_LEN) / (FFT_LEN - 1))

def extract_features_c_style(file_path):
    """
    Simulates the exact processing pipeline of the STM32.
    """
    try:
        # 1. Load Audio
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        audio, _ = librosa.effects.trim(audio)
        
        # 2. SCALE TO INT16 (Crucial Step!)
        # The C code operates on int16_t PCM audio.
        # Librosa gives floats -1.0 to 1.0. We must scale up.
        audio = audio * 32767.0
        
        # 3. Framing
        num_frames = (len(audio) - FFT_LEN) // HOP_LEN + 1
        if num_frames < 1: return None
        
        mfcc_accum = np.zeros(NUM_DCT_COEFFS)
        
        for i in range(num_frames):
            start = i * HOP_LEN
            frame = audio[start : start + FFT_LEN]
            
            # Match C: Windowing
            frame = frame * WINDOW
            
            # Match C: Real FFT -> Power Spectrum
            fft_out = np.fft.rfft(frame)
            power_spec = (np.abs(fft_out) ** 2)
            
            # Match C: Mel Energies + Log
            # 1e-12 matches the epsilon in typical implementations
            mel_energies = np.dot(MEL_FILTERS, power_spec)
            mel_energies = np.maximum(mel_energies, 1e-12)
            log_mel = np.log(mel_energies) 
            
            # Match C: DCT
            mfccs = np.dot(DCT_MAT, log_mel)
            
            mfcc_accum += mfccs
            
        # 4. Average
        return mfcc_accum / num_frames

    except Exception as e:
        print(f"Err: {e}")
        return None

# ==========================================
# MAIN EXECUTION
# ==========================================
def main():
    os.makedirs(EXPORT_DIR, exist_ok=True)
    wav_files = glob.glob(os.path.join(DATASET_PATH, "*.wav"))
    
    if not wav_files:
        print("❌ No recordings found.")
        return

    print(f"Processing {len(wav_files)} files using C-compatible logic...")
    
    X, y = [], []
    for i, f in enumerate(wav_files):
        label = int(os.path.basename(f).split('_')[0])
        feat = extract_features_c_style(f)
        if feat is not None:
            X.append(feat)
            y.append(label)
        if i % 100 == 0: print(f"\r  Progress: {i}/{len(wav_files)}", end="")
    print("")

    X = np.array(X).astype(np.float32)
    y = np.array(y).astype(int)
    
    # --- SPLIT TRAIN (For Flash) / TEST (For PC Validation) ---
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # --- SUBSAMPLE TRAINING DATA (Flash Limit) ---
    if len(X_train) > NUM_SAMPLES_TO_KEEP:
        print(f"Reducing Train set from {len(X_train)} to {NUM_SAMPLES_TO_KEEP} for Flash memory...")
        from sklearn.utils import shuffle
        X_train, y_train = shuffle(X_train, y_train, random_state=42)
        X_train = X_train[:NUM_SAMPLES_TO_KEEP]
        y_train = y_train[:NUM_SAMPLES_TO_KEEP]

    # --- TRAIN & EXPORT ---
    print("\nTraining kNN...")
    knn = KNNClassifier(n_neighbors=3)
    knn.train(X_train, y_train) # Train only on the subset going to Flash

    # --- VALIDATE ---
    # We validate on X_test (which the board has never seen)
    from sklearn.neighbors import KNeighborsClassifier
    sk = KNeighborsClassifier(n_neighbors=3)
    sk.fit(X_train, y_train)
    acc = sk.score(X_test, y_test)
    print("="*40)
    print(f"📊 EXPECTED ACCURACY ON BOARD: {acc*100:.2f}%")
    print("="*40)

    # Export
    out_path = os.path.join(EXPORT_DIR, EXPORT_NAME)
    knn.export(out_path)
    
    # Append Strings
    unique_classes = sorted(np.unique(y))
    c_strings = ", ".join([f'"{c}"' for c in unique_classes])
    
    with open(out_path + ".h", "a") as f:
        f.write(f"\n#define NUM_UNIQUE_CLASSES {len(unique_classes)}\n")
        f.write(f"extern const char* CLASS_NAMES[NUM_UNIQUE_CLASSES];\n")
    with open(out_path + ".c", "a") as f:
        f.write(f"\nconst char* CLASS_NAMES[NUM_UNIQUE_CLASSES] = {{ {c_strings} }};\n")

    print(f"✅ Config saved to {out_path}.h/.c")

if __name__ == "__main__":
    main()

Processing 2700 files using C-compatible logic...
  Progress: 2600/2700
Reducing Train set from 2160 to 500 for Flash memory...

Training kNN...
📊 EXPECTED ACCURACY ON BOARD: 74.63%
✅ Config saved to exported_models\knn_mfcc_config.h/.c
